In [32]:
import numpy as np
import pandas as pd
import io, zipfile, urllib.request
from scipy import stats
from math import *
from scipy.special import digamma, polygamma

In [ ]:
###### APPLICATION ########

In [33]:

#### Definiton of the sample correlation matrix ####

# 1. URL to Fama-French 48 Industry Portfolios (Daily)
url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/48_Industry_Portfolios_daily_CSV.zip"

# 2. Download and read dynamically (handles any filename inside zip)
req = urllib.request.urlopen(url)
with zipfile.ZipFile(io.BytesIO(req.read())) as z:
    # Get the exact filename dynamically from the zip contents
    filename = z.namelist()[0]
    df = pd.read_csv(z.open(filename), skiprows=9, index_col=0)

# 3. Clean string indices and missing value codes
df.index = df.index.astype(str).str.strip()

# Convert to numeric, coercion replaces missing code (-99.99 or -999) with NaN
df = df.apply(pd.to_numeric, errors='coerce').dropna()

# 4. Filter for valid industry portfolio columns (drop non-numeric rows/headers if present)
df = df.select_dtypes(include=[np.number])

# Take a short window of n = 30 days to enforce p > n (p = 48)
X_subset = df.iloc[-30:] 

# Sample correlation matrix
R = X_subset.corr()

# Number of variables
p = X_subset.shape[1]

# Sample size
n = X_subset.shape[0]


C:\Users\fjm\AppData\Local\Temp\ipykernel_9432\524889346.py:11: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(z.open(filename), skiprows=9, index_col=0)


In [34]:


def compute_log_independence_test(R, n):
    """
    Computes the test statistic T and its asymptotic Gamma-approximated p-value 
    for complete independence in high-dimensional correlation matrices.
    
    Parameters:
    -----------
    R : pandas.DataFrame or numpy.ndarray
        Sample correlation matrix of dimension (p, p).
    n : int
        Sample size (number of observations/patients/days).
        
    Returns:
    --------
    results : dict
        Dictionary containing the observed statistic (T), p-value, 
        theoretical mean, theoretical variance, and fitted Gamma parameters.
    """
    # Convert pandas DataFrame to numpy array if necessary
    if hasattr(R, 'to_numpy'):
        R_mat = R.to_numpy()
    else:
        R_mat = np.asarray(R)
        
    p = R_mat.shape[0]
    
    # 1. Compute Test Statistic T = sum_{i < j} -log(1 - r_ij^2)
    # Extract strictly upper-triangular elements
    triu_indices = np.triu_indices(p, k=1)
    r_squared_pairs = R_mat[triu_indices]**2
    
    # Clip to avoid log(0) or small negative numbers due to floating point precision
    AAA = np.clip(1.0 - r_squared_pairs, a_min=1e-15, a_max=1.0)
    
    # Numerically stable sum of logs (replaces -log(prod(AAA)))
    tobs = np.sum(-np.log(AAA))
    
    # 2. Moment Calculations under H0
    m = p * (p - 1) / 2
    
    delta = digamma((n - 1) / 2) - digamma((n - 2) / 2)
    m1 = m * delta  # Theoretical mean E[T]
    
    trigamma = polygamma(1, (n - 2) / 2) - polygamma(1, (n - 1) / 2)
    m2 = m * trigamma + (m * delta)**2  # Second raw moment E[T^2]
    
    var_T = m2 - m1**2  # Variance Var(T)
    
    # 3. Method of Moments Gamma Distribution Matching
    # Shape parameter: r = E[T]^2 / Var(T)
    r = -m1**2 / (m1**2 - m2)
    
    # Scale parameter: lambda = Var(T) / E[T]
    lamb = (m2 / m1) - m1
    
    # 4. p-value calculation via Survival Function for floating-point accuracy
    pval = stats.gamma.sf(tobs, a=r, scale=lamb)
    
    return {
        'T_obs': tobs,
        'p_value': pval,
        'mean_null': m1,
        'var_null': var_T,
        'gamma_shape': r,
        'gamma_scale': lamb
    }

In [35]:
compute_log_independence_test(R, n)

{'T_obs': 202.8205250684466,
 'p_value': 0.0,
 'mean_null': 41.004644411710146,
 'var_null': 2.9801901139458096,
 'gamma_shape': 564.1857730695652,
 'gamma_scale': 0.07267933076124677}